# Frontend API Stability and Robustness Testing

This notebook validates frontend HTTP API interactions with the FastAPI backend. It includes health checks, single-endpoint functional validation, multi-threaded stress testing, error handling, and test log analysis.

## 1. Import Test Libraries and Frontend Environment

Load common test libraries such as requests, concurrent.futures, and time, and configure the frontend API base URL for this notebook.

In [ ]:
import os
import time
import json
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests

API_BASE_URL = os.getenv("API_BASE_URL", "http://127.0.0.1:8000").rstrip("/")
API_PREFIX = os.getenv("API_PREFIX", "/api/cost-living").rstrip("/")
TIMEOUT_SECONDS = int(os.getenv("API_TIMEOUT_SECONDS", "60"))
PREFLIGHT_TIMEOUT_SECONDS = int(os.getenv("API_PREFLIGHT_TIMEOUT_SECONDS", "15"))
ERROR_TIMEOUT_SECONDS = int(os.getenv("API_ERROR_TIMEOUT_SECONDS", "15"))
MAX_WORKERS = int(os.getenv("API_MAX_WORKERS", "2"))
RUN_EXTENDED = os.getenv("RUN_EXTENDED", "0") == "1"
RUN_STRESS = os.getenv("RUN_STRESS", "0") == "1"

print("Current API_BASE_URL for this test notebook = %s" % API_BASE_URL)
print("Current API_PREFIX for this test notebook = %s" % API_PREFIX)
print("Default timeout = %s seconds" % TIMEOUT_SECONDS)


## 2. Define Helper Functions for API Requests

Implement reusable helper functions for calling API endpoints, checking statuses, and fetching data.

In [ ]:
ENDPOINTS = [
    "/api/health",
    "/api/pipeline/status",
    "/api/stats/overview",
    "/api/trends/documents",
    "/api/categories/counts",
    "/api/categories/sentiment",
    "/api/categories/share",
    "/api/data-quality/summary",
    "/api/data-quality/comparison",
    "/api/media/coverage",
    "/api/platforms/categories",
    "/api/trends/categories",
    "/api/trends/sentiment",
    "/api/official/comparison",
    "/api/categories/yoy-change",
    "/api/categories/volatility",
    "/api/categories/keywords",
    "/api/logs/errors",
]

COMMON_PARAMS = {
    "source_group": "social",
    "quality": "clean",
    "period": "month",
    "start": "2026-01-01",
    "end": "2026-03-31",
}

def api_get(path, params=None, timeout=None):
    request_path = API_PREFIX + path[4:] if path.startswith("/api/") else API_PREFIX + path
    url = f"{API_BASE_URL}{request_path}"
    params = {k: v for k, v in (params or {}).items() if v not in (None, "")}
    request_timeout = timeout or TIMEOUT_SECONDS
    start = time.monotonic()
    try:
        response = requests.get(url, params=params, timeout=request_timeout)
    except requests.RequestException as exc:
        duration = time.monotonic() - start
        return {
            "path": path,
            "url": url,
            "params": params,
            "status_code": None,
            "ok": False,
            "duration": duration,
            "body": {"error": f"{type(exc).__name__}: {exc}"},
        }
    duration = time.monotonic() - start
    try:
        body = response.json()
    except ValueError:
        body = response.text
    return {
        "path": path,
        "url": url,
        "params": params,
        "status_code": response.status_code,
        "ok": response.ok,
        "duration": duration,
        "body": body,
    }

EXPECTED_RESPONSE_KEYS = {
    "/api/health": {"service", "status", "elasticsearch"},
    "/api/pipeline/status": {"raw_documents", "processed_documents", "official_indicators"},
    "/api/stats/overview": {"total_documents", "categories", "platforms"},
    "/api/data-quality/summary": {"total_documents", "clean_documents", "quality_flags"},
    "/api/data-quality/comparison": {"rows", "quality_modes"},
    "/api/media/coverage": {"rows", "source_group"},
}

def validate_response(result):
    if not result["ok"]:
        body = result.get("body")
        if isinstance(body, dict) and body.get("error"):
            return False, body["error"][:160]
        return False, f"HTTP {result['status_code']}"
    body = result["body"]
    if not isinstance(body, dict):
        return False, "response body is not a JSON object"
    expected_keys = EXPECTED_RESPONSE_KEYS.get(result["path"])
    if expected_keys and not expected_keys.issubset(body.keys()):
        return False, "unexpected response shape: expected keys %s" % sorted(expected_keys)
    if "rows" in body and not isinstance(body["rows"], list):
        return False, "rows field is not a list"
    return True, "OK"

def run_smoke_test(path, params=None, timeout=None):
    result = api_get(path, params=params, timeout=timeout)
    ok, reason = validate_response(result)
    result["validated"] = ok
    result["validation_reason"] = reason
    return result

def run_requests_concurrently(requests_list, max_workers=MAX_WORKERS):
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_request = {
            executor.submit(api_get, path, params): (path, params)
            for path, params in requests_list
        }
        for future in as_completed(future_to_request):
            try:
                result = future.result()
                ok, reason = validate_response(result)
                result["validated"] = ok
                result["validation_reason"] = reason
                results.append(result)
            except Exception as exc:
                path, params = future_to_request[future]
                results.append({
                    "path": path,
                    "url": None,
                    "params": params,
                    "status_code": None,
                    "ok": False,
                    "duration": None,
                    "body": str(exc),
                    "validated": False,
                    "validation_reason": "exception during request",
                })
    return results

def summarize_body(body):
    if isinstance(body, dict):
        if "error" in body:
            return body["error"]
        parts = []
        if "rows" in body and isinstance(body["rows"], list):
            parts.append("rows=%d" % len(body["rows"]))
        for key in ["status", "service", "total_documents", "raw_documents", "processed_documents", "official_indicators", "detail"]:
            if key in body:
                parts.append("%s=%s" % (key, body[key]))
        return "; ".join(parts) or "keys=" + ",".join(sorted(body.keys())[:8])
    return str(body)[:240]

def format_report(results):
    df = pd.DataFrame(results)
    if "duration" in df.columns:
        df["duration"] = pd.to_numeric(df["duration"], errors="coerce").round(3)
    if "body" in df.columns:
        df["body_summary"] = df["body"].map(summarize_body)
        df = df.drop(columns=["body"])
    return df


## 3. Check API Status

Use `GET /api/health` and `GET /api/pipeline/status` to verify backend availability. The frontend does not inspect infrastructure internals directly; health endpoints indicate service availability.

In [ ]:
health_result = run_smoke_test("/api/health", timeout=PREFLIGHT_TIMEOUT_SECONDS)

if not health_result["ok"]:
    status_df = format_report([health_result])
    display(status_df)
    raise RuntimeError(
        "API preflight failed. Check API_BASE_URL/API_PREFIX and restart kubectl port-forward before running the rest of this notebook."
    )

pipeline_result = run_smoke_test("/api/pipeline/status")
status_df = format_report([health_result, pipeline_result])
display(status_df)

print("API health check summary:")
print("- /api/health returned %s" % ("OK" if health_result["ok"] else health_result["body"]))
print("- /api/pipeline/status returned %s" % ("OK" if pipeline_result["ok"] else pipeline_result["body"]))


## 4. Frontend Single-Request Functional Validation

Verify that key frontend endpoints return expected results, and check common parameter combinations, response structure, and data ranges.

In [ ]:
core_tests = [
    ("/api/stats/overview", COMMON_PARAMS),
    ("/api/trends/documents", {**COMMON_PARAMS, "period": "day", "time_field": "created_at"}),
    ("/api/categories/counts", COMMON_PARAMS),
    ("/api/categories/sentiment", COMMON_PARAMS),
    ("/api/trends/sentiment", COMMON_PARAMS),
    ("/api/official/comparison", COMMON_PARAMS),
]

extended_tests = [
    ("/api/data-quality/summary", COMMON_PARAMS),
    ("/api/data-quality/comparison", COMMON_PARAMS),
    ("/api/media/coverage", {"period": "month", "quality": "all"}),
    ("/api/platforms/categories", COMMON_PARAMS),
    ("/api/trends/categories", COMMON_PARAMS),
    ("/api/categories/yoy-change", COMMON_PARAMS),
    ("/api/categories/volatility", COMMON_PARAMS),
    ("/api/categories/keywords", {**COMMON_PARAMS, "limit": 10}),
    ("/api/logs/errors", {"size": 20}),
]

single_tests = core_tests + (extended_tests if RUN_EXTENDED else [])
if not RUN_EXTENDED:
    print("Extended endpoint tests skipped by default. Set RUN_EXTENDED=1 to include heavier aggregations.")

single_results = [run_smoke_test(path, params) for path, params in single_tests]
single_df = format_report(single_results)
display(single_df)
print("Single-request functional validation complete: %d requests, %d successful." % (len(single_results), sum(1 for r in single_results if r['ok'])))


## 5. Frontend Multi-threaded Concurrent Access Stress Test

Simulate multiple frontend users accessing the API simultaneously, and collect success rates, response times, and failure details.

In [ ]:
if RUN_STRESS:
    stress_requests = []
    for _ in range(2):
        stress_requests.append(("/api/health", None))
        stress_requests.append(("/api/pipeline/status", None))
        stress_requests.append(("/api/stats/overview", COMMON_PARAMS))
        stress_requests.append(("/api/trends/documents", {**COMMON_PARAMS, "period": "month"}))
        stress_requests.append(("/api/categories/counts", COMMON_PARAMS))
        stress_requests.append(("/api/categories/sentiment", COMMON_PARAMS))

    stress_results = run_requests_concurrently(stress_requests, max_workers=MAX_WORKERS)
    stress_df = format_report(stress_results)
    summary = {
        "total_requests": len(stress_results),
        "success_count": sum(1 for r in stress_results if r['ok']),
        "failure_count": sum(1 for r in stress_results if not r['ok']),
        "average_duration": stress_df['duration'].dropna().mean(),
        "max_duration": stress_df['duration'].dropna().max(),
    }
    display(stress_df.groupby('path').agg({
        'ok': ['mean', 'count'],
        'duration': ['mean', 'max'],
        'status_code': lambda x: Counter(x.dropna()).most_common(),
    }))
else:
    stress_results = []
    stress_df = pd.DataFrame(columns=["path", "params", "status_code", "ok", "duration", "body"])
    summary = {"skipped": True, "reason": "Set RUN_STRESS=1 to run the bounded burst test."}

print("Stress test results: %s" % json.dumps(summary, default=str, indent=2))

## 6. Error Handling and Response Stability Validation

Test invalid parameters, timeouts, and backend error responses to verify that the frontend can correctly identify failures under abnormal conditions.

In [ ]:
error_tests = [
    ("/api/trends/documents", {**COMMON_PARAMS, "period": "invalid-period"}),
    ("/api/categories/counts", {**COMMON_PARAMS, "source_group": "wrong-group"}),
    ("/api/categories/keywords", {"limit": "bad"}),
]

error_results = []
for path, params in error_tests:
    result = api_get(path, params=params, timeout=ERROR_TIMEOUT_SECONDS)
    result["validated"] = result["status_code"] in {400, 422}
    result["validation_reason"] = "expected fast 4xx for invalid input"
    error_results.append(result)

error_df = format_report(error_results)
display(error_df)
print("Error response validation complete: %d invalid requests checked." % len(error_results))


## 7. Test Result Logging and Analysis

Aggregate the test results and generate a table for analysis. Export to CSV for future comparison.

In [ ]:
report = pd.concat([
    status_df.assign(stage='health'),
    single_df.assign(stage='single'),
    stress_df.assign(stage='stress'),
    error_df.assign(stage='error'),
], ignore_index=True, sort=False)
report['timestamp'] = pd.Timestamp.now()
display(report.head(20))

output_dir = Path.cwd() if Path.cwd().name == 'frontend' else Path.cwd() / 'frontend'
report_path = output_dir / 'frontend_api_robustness_report.csv'
report.to_csv(report_path, index=False)
print("Test report generated: %s" % report_path)
